# FlashAttention: Reordering Data Movement

> Mixed precision reduces bytes per value, but Attention still creates intermediates that grow quadratically with sequence length. At 4,096 tokens, the score matrix contains about 16.8 million values. A standard implementation writes it to HBM, then reads it again for Softmax and weighted summation.
>
> **FlashAttention** primarily reduces data movement between HBM and on-chip SRAM rather than the number of multiplications. **Online Softmax** maintains a running maximum and denominator across blocks. **Tiling** keeps small Q/K/V blocks in SRAM and accumulates output before writing it back.
>
> The computation order changes, but the mathematical Attention result remains the same. Understanding why requires joining numerical stability, block state, and the GPU memory hierarchy in one data flow.


A 4,096-token sequence produces a `4096 x 4096` score matrix with about 16.8 million entries. Standard Attention writes this intermediate to HBM, reads it for Softmax, and reads or writes another matrix before multiplying by $V$.

FlashAttention preserves the mathematical definition but changes execution order. It loads Q, K, and V tiles into fast on-chip SRAM and uses Online Softmax to maintain maxima, denominators, and output accumulators. The full $N\times N$ matrix is never written to HBM.


In [ ]:
# This appendix uses only torch and numpy, no high-level attention API
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(f"torch: {torch.__version__}")


## 1. GPU Memory Hierarchy

SRAM may offer roughly an order of magnitude more bandwidth than HBM but only a tiny fraction of its capacity. Matrix arithmetic can therefore be faster than moving operands into and out of the compute units. Such workloads are **memory-bound**, and Attention is a prominent example.

FlashAttention asks how to retain useful intermediates in SRAM and reduce round trips to HBM.


### Data Movement in Standard Attention

The maximum and Softmax do not fundamentally require a complete score matrix to be stored. They can be updated while scanning. Standard staged implementations nevertheless materialize $N\times N$ intermediates between kernels, causing repeated HBM traffic. The next calculation quantifies this scale.


In [ ]:
# Memory footprint of the standard attention matrix at different sequence lengths (FP16, 2 bytes/element)
def attn_matrix_bytes(seq, head_dim=128, n_heads=32, bytes_per_elem=2):
    # Intermediate matrices written to HBM per forward pass: S (attention scores) and P (after softmax)
    # One copy per head, per batch
    elems = n_heads * seq * seq
    return elems * bytes_per_elem

def fmt(b):
    if b >= 1024**3:
        return f"{b/1024**3:.2f} GB"
    if b >= 1024**2:
        return f"{b/1024**2:.1f} MB"
    return f"{b/1024:.1f} KB"

print(f"{'seq len':>8}  {'one S/P matrix':>15}  {'5 read/writes total':>20}")
print("-" * 50)
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    single = attn_matrix_bytes(seq)
    # S written 1x, read 2x (for max, for softmax); P written 1x, read 1x
    traffic = single * 5
    print(f"{seq:>8d}  {fmt(single):>15}  {fmt(traffic):>15}")

print()
print("At seq=32768, moving the attention matrices alone requires on the order of 1 TB of data.")
print("The A100's HBM bandwidth is 2 TB/s, so the transfer alone takes about 0.5 seconds — far slower than the compute itself.")


## 2. Numerical Stability of Softmax

The exponential grows rapidly; in FP32, values near $e^{88}$ overflow. Softmax computes exponentials in both numerator and denominator, so one logit near 100 can turn an entire row into Inf/NaN.

Attention logits are dot products of $d$-dimensional Q and K vectors, and large values are plausible at $d=128$. Stable computation is therefore necessary, not optional.


In [ ]:
# Demonstrate the overflow threshold of exp under FP32
import torch

x = torch.tensor([80.0, 85.0, 88.0, 90.0, 100.0])
print("x       exp(x)")
for xi in x:
    print(f"{xi.item():>6.1f}  {torch.exp(xi).item():>15.3e}")

print()
print("Under FP32, exp(89) is already close to inf.")

# Simulate the magnitude of an attention score
d_head = 128
q = torch.randn(d_head)
k = torch.randn(d_head)
score = (q * k).sum() / math.sqrt(d_head)
print(f"With random initialization and d=128, the attention score has a scale around ±{abs(score.item()):.1f}")
print("But late in training, q and k norms can grow 5-10x, pushing the score above 80.")


Subtracting a constant $c$ from every logit multiplies numerator and denominator by the same $e^{-c}$, leaving Softmax unchanged. Choosing $c=x_{max}$ makes the largest exponential $e^0=1$ and every other exponential at most 1.

Stable Softmax first finds the maximum, then calculates $e^{x_i-x_{max}}/\sum_j e^{x_j-x_{max}}$. It prevents overflow but appears to require two passes. Online Softmax removes that requirement.


In [ ]:
def naive_softmax(x):
    """Direct implementation, will overflow"""
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum()

def stable_softmax(x):
    """Subtract max for numerical stability"""
    m = x.max()
    exp_x = torch.exp(x - m)
    return exp_x / exp_x.sum()

# Construct an input that makes the naive version blow up
x = torch.tensor([1.0, 2.0, 3.0]) * 50  # max is 150
print('"Input:"', x.tolist())
print("naive softmax:", naive_softmax(x).tolist())
print("stable softmax:", stable_softmax(x).tolist())

print()
print("In the naive version, exp(150) overflows to infinity and the final result becomes NaN.")
print("In the stable version, max=150 is subtracted away, so the largest exponent is exp(0)=1 — safe.")


After subtracting the maximum, the sum contains at least one $e^0=1$, so its logarithm is well-defined. This logsumexp trick underlies framework implementations of `log_softmax`. Online Softmax maintains the same maximum and normalized sum incrementally.


In [ ]:
def naive_logsumexp(x):
    """Direct log(sum(exp(x))), will overflow"""
    return torch.log(torch.exp(x).sum())

def stable_logsumexp(x):
    """logsumexp trick"""
    m = x.max()
    return m + torch.log(torch.exp(x - m).sum())

x = torch.tensor([100.0, 101.0, 102.0])
print('"Input:"', x.tolist())
print("naive logsumexp:", naive_logsumexp(x).item())
print("stable logsumexp:", stable_logsumexp(x).item())
print("official torch: ", torch.logsumexp(x, dim=0).item())

print()
print("These three should be exactly equal (except the naive one blows up to nan).")
print("This trick will reappear repeatedly in online softmax —")
print("FlashAttention's running variables are essentially a streaming version of logsumexp.")


## 3. Online Softmax

A two-pass stable Softmax reads the data once for the maximum and again for the denominator. Online Softmax instead maintains a running maximum and a denominator expressed relative to that maximum. When a larger value appears, it rescales the old accumulator to the new reference.

This gives FlashAttention its first core technique: **update the maximum and denominator together, rescaling prior work whenever the reference maximum changes**.


### 3.1 Two-Pass and One-Pass Algorithms

The one-pass ledger contains only:

$$m_k=\max(x_1,\ldots,x_k),\qquad d_k=\sum_{j=1}^k e^{x_j-m_k}$$

If the maximum changes from $m_k$ to $m_{k+1}$, every previous term converts by the common factor $e^{m_k-m_{k+1}}$. We can multiply the whole old denominator once rather than revisit each earlier element. If the maximum is unchanged, the factor is 1.


### 3.2 Online Softmax by Hand

For $x=[3,1,2]$, the first item sets the running maximum to 3 and denominator to 1. Later maxima remain 3, so each new step adds $e^{x_i-3}$. The algorithm reads each value once and keeps only two running scalars.


In [ ]:
def online_softmax(x):
    """Single-pass softmax, maintains running (m, d)"""
    m = float('-inf')   # running max
    d = 0.0             # running denominator sum exp(x_i - m)

    for xi in x:
        m_new = max(m, xi.item())
        # rescale the old sum and add the new term
        d = d * math.exp(m - m_new) + math.exp(xi.item() - m_new)
        m = m_new

    # Second pass: use the final (m, d) to compute each position's softmax
    # FlashAttention folds this step into the forward pass via tiling
    return torch.tensor([math.exp(xi.item() - m) / d for xi in x])

x = torch.tensor([3.0, 1.0, 2.0])
print('"Input:"', x.tolist())
print("online softmax:    ", online_softmax(x).tolist())
print("stable softmax:    ", stable_softmax(x).tolist())
print("official torch softmax:", torch.softmax(x, dim=0).tolist())

print()
print("All three are numerically identical — the online flow introduces no approximation.")


## 4. From Softmax to the Attention Output

The final attention output is not softmax itself, but $o = \sum_i p_i v_i$, where $p_i$ is the softmax weight and $v_i$ is the value vector. Expanding $p_i$:

$$
o = \sum_i \frac{e^{x_i}}{\sum_j e^{x_j}} v_i = \frac{\sum_i e^{x_i} v_i}{\sum_i e^{x_i}}
$$

Multiply numerator and denominator by $e^{-x_\max}$ to cancel the overflow:

$$
o = \frac{\sum_i e^{x_i - x_\max} v_i}{\sum_i e^{x_i - x_\max}}
$$

Here $x_i = q \cdot k_i / \sqrt{d}$ is the dot product of the query with the $i$-th key, and $v_i$ is the $i$-th value. In addition to $(m, d)$, we maintain a running output $o_k = \sum_{i=1}^k e^{x_i - m_k} v_i$.

The update rule is perfectly symmetric to that of $d$, because both are essentially "a weighted sum of $e^{x_i - m}$":

$$
o_{k+1} = o_k \cdot e^{m_k - m_{k+1}} + e^{x_{k+1} - m_{k+1}} v_{k+1}
$$

After scanning all $N$ keys, the true attention output is $o_N / d_N$.


### 4.1 Attention Output by Hand

The online Attention accumulator $o$ follows the same rescaling rule as denominator $d$, except each new exponential weight is multiplied by its corresponding $v_i$. The final normalized output is $o/d$.


In [ ]:
def online_attention(q, K, V):
    """
    One-pass attention, processing each query independently.

    Parameters:
        q: one query vector, shape [d]
        K: all keys, shape [N, d]
        V: all values, shape [N, d]
    Returns:
        attention output, shape [d]
    """
    d_head = q.shape[0]
    N = K.shape[0]

    m = float('-inf')       # running maximum
    d = 0.0                 # running denominator
    o = torch.zeros_like(q) # running output

    for i in range(N):
        # Dot product between this key and the query; divide by sqrt(d) for stable gradients
        x_i = torch.dot(q, K[i]).item() / math.sqrt(d_head)
        v_i = V[i]

        m_new = max(m, x_i)
        rescale = math.exp(m - m_new)

        d = d * rescale + math.exp(x_i - m_new)
        o = o * rescale + math.exp(x_i - m_new) * v_i
        m = m_new

    return o / d

# Hand calculation: d=1, q=1, k=[3,1,2], v=[10,20,30]
q = torch.tensor([1.0])
K = torch.tensor([[3.0], [1.0], [2.0]])
V = torch.tensor([[10.0], [20.0], [30.0]])
print("online attention output:", online_attention(q, K, V).item())
print("hand-calculated result: 15.80")

# Compare with the standard implementation
def standard_attention(q, K, V):
    d_head = q.shape[0]
    scores = (K @ q) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=0)
    return weights @ V

print("standard attention output:", standard_attention(q, K, V).item())


## 5. Tiling

A token-by-token CPU loop is mathematically useful but unsuitable for a massively parallel GPU. FlashAttention divides Q, K, and V into blocks sized to fit SRAM. Blocks can be processed with matrix operations, and many blocks run in parallel.

Intermediate score and probability tiles $S$ and $P$ are only $B\times B$ and remain in SRAM. Each tile is consumed and overwritten rather than written to HBM.

### 5.2 Why Tiling Is Equivalent

For separate key blocks, calculate local states $(m_1,d_1,o_1)$ and $(m_2,d_2,o_2)$, choose global $m=\max(m_1,m_2)$, rescale both states to $m$, and add them. This equals scanning all keys at once. Online updates depend only on $(m,d,o)$, so block boundaries do not change the result.


### 5.3 Choosing Block Size

Each block must fit Q, K, V tiles ($3Bd$ elements) plus score and probability tiles ($2B^2$), for roughly $3Bd+2B^2$ elements of SRAM.

Small $B$ increases loop and launch overhead. Large $B$ exceeds SRAM and spills to HBM, defeating the optimization. Production kernels choose $B$ from GPU architecture and head dimension rather than exposing it as a typical user parameter.


## 6. Exact Reordering, Not Approximation

Sparse Attention skips query–key pairs, and Linear Attention replaces softmax with a decomposable feature mapping. Both change the mathematical output.

FlashAttention changes only evaluation order. If all intermediates were retained, it would produce the same result as dense Attention up to floating-point rounding. This is why it can replace standard Attention without changing the training objective or model architecture.


## 7. A Simplified PyTorch Implementation

Next, we implement a teaching version of FlashAttention in pure PyTorch. **It does not aim for performance** (Python loops are far slower than CUDA); it aims only for mathematical correctness — so the reader can see exactly what every step of online softmax plus tiling is doing.

Implementation notes:

- Single head, batch=1 for simplicity (multiple heads are just an outer loop)
- Outer loop over Q blocks, inner loop over K/V blocks
- Use `torch.einsum` or `@` for the block-level matrix multiplies
- Explicitly maintain the three running variables $(m, d, o)$


In [ ]:
def flash_attention_forward(Q, K, V, block_size=4, causal=False):
    """
    Teaching implementation of FlashAttention forward, one head and batch=1.

    Parameters:
        Q: shape [N, d]
        K: shape [N, d]
        V: shape [N, d]
        block_size: block size for Q/K/V
        causal: whether to apply a causal lower-triangular mask
    Returns:
        O: shape [N, d]
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # This simplified teaching version keeps the output and running variables in HBM
    # Real FlashAttention keeps them in SRAM
    O = torch.zeros(N, d)

    # Outer loop: process each Q block independently
    n_blocks = (N + block_size - 1) // block_size
    for i in range(n_blocks):
        q_start = i * block_size
        q_end = min(q_start + block_size, N)
        Qi = Q[q_start:q_end] * scale  # [B_q, d], apply scale in advance

        # Running variables for this Q block
        B_q = q_end - q_start
        m_i = torch.full((B_q,), float('-inf'))
        d_i = torch.zeros(B_q)
        o_i = torch.zeros(B_q, d)

        # Inner loop: visit every K/V block
        for j in range(n_blocks):
            k_start = j * block_size
            k_end = min(k_start + block_size, N)
            Kj = K[k_start:k_end]  # [B_k, d]
            Vj = V[k_start:k_end]  # [B_k, d]

            # Compute this block's attention scores, which exist only in SRAM
            S = Qi @ Kj.transpose(-2, -1)  # [B_q, B_k]

            # Causal mask: set entries where query position < key position to -inf
            if causal:
                q_idx = torch.arange(q_start, q_end)[:, None]
                k_idx = torch.arange(k_start, k_end)[None, :]
                mask = k_idx > q_idx
                S = S.masked_fill(mask, float('-inf'))

            # Online softmax update
            m_block = S.max(dim=-1).values      # [B_q]
            m_new = torch.maximum(m_i, m_block)

            # Rescale the old accumulator
            alpha = torch.exp(m_i - m_new)      # [B_q]
            # Exponentials for the new block; subtract the new maximum for numerical stability
            P = torch.exp(S - m_new[:, None])   # [B_q, B_k]

            d_i = d_i * alpha + P.sum(dim=-1)
            o_i = o_i * alpha[:, None] + P @ Vj
            m_i = m_new

        # Final output for this Q block
        O[q_start:q_end] = o_i / d_i[:, None]

    return O

# Small example: seq=8 and d=4, friendly for hand calculation
torch.manual_seed(42)
N, d = 8, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

O_flash = flash_attention_forward(Q, K, V, block_size=4)
print("FlashAttention output (first 3 rows):")
print(O_flash[:3])


## 8. Numerical Equivalence Check

Compare a from-scratch dense Attention that explicitly builds $N\times N$ scores with PyTorch `scaled_dot_product_attention`, which may select a FlashAttention backend. Differences near $10^{-6}$ are floating-point rounding evidence of mathematical equivalence.


In [ ]:
def standard_attention(Q, K, V, causal=False):
    """Standard attention, explicitly constructs the N×N matrix"""
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    S = (Q @ K.T) * scale  # [N, N]
    if causal:
        mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
        S = S.masked_fill(mask, float('-inf'))
    P = torch.softmax(S, dim=-1)
    return P @ V

# Non-causal case
O_std = standard_attention(Q, K, V, causal=False)
O_sdpa = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=False
)[0]

print("=== Non-causal case ===")
print(f"Flash vs standard   max diff: {(O_flash - O_std).abs().max().item():.2e}")
print(f"Flash vs SDPA       max diff: {(O_flash - O_sdpa).abs().max().item():.2e}")
print(f"Standard vs SDPA    max diff: {(O_std - O_sdpa).abs().max().item():.2e}")

# Causal case
O_flash_causal = flash_attention_forward(Q, K, V, block_size=4, causal=True)
O_std_causal = standard_attention(Q, K, V, causal=True)
O_sdpa_causal = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=True
)[0]

print()
print("=== Causal case ===")
print(f"Flash vs standard   max diff: {(O_flash_causal - O_std_causal).abs().max().item():.2e}")
print(f"Flash vs SDPA       max diff: {(O_flash_causal - O_sdpa_causal).abs().max().item():.2e}")

print()
print("All differences are at the 1e-6 level — within floating point error, so FlashAttention is mathematically equivalent.")


In [ ]:
# Larger test: seq=256, d=64, to confirm accuracy on longer sequences
torch.manual_seed(0)
N, d = 256, 64
Q_big = torch.randn(N, d)
K_big = torch.randn(N, d)
V_big = torch.randn(N, d)

O_flash_big = flash_attention_forward(Q_big, K_big, V_big, block_size=64)
O_sdpa_big = F.scaled_dot_product_attention(
    Q_big[None], K_big[None], V_big[None]
)[0]

max_diff = (O_flash_big - O_sdpa_big).abs().max().item()
mean_diff = (O_flash_big - O_sdpa_big).abs().mean().item()
print(f"seq=256, d=64:")
print(f"  max diff:  {max_diff:.2e}")
print(f"  mean diff: {mean_diff:.2e}")
print(f"  passes 1e-5 threshold: {max_diff < 1e-5}")

# Also check the effect of block size on the result (should have no effect)
print()
print("Effect of block size on numerical correctness (all should be < 1e-5):")
for bs in [16, 32, 64, 128]:
    O_bs = flash_attention_forward(Q_big, K_big, V_big, block_size=bs)
    diff = (O_bs - O_sdpa_big).abs().max().item()
    print(f"  block_size={bs:>3d}: max diff {diff:.2e}")


## 9. FlashAttention with MoE

**Attention itself does not change in an MoE model; the challenge is how routed token batches are organized.** Expert routing creates variable-length token groups, so efficient implementations either pad groups to uniform shapes or use variable-length kernels and metadata.

### 9.2 Two Implementation Approaches


In [ ]:
# Quantify the waste of padding under MoE
def moe_padding_waste(B, S, E, k, n_heads, d):
    """Compute the effective compute ratio of MoE under padding mode"""
    # Average number of tokens each expert receives
    avg_tokens_per_expert = B * S * k / E
    # Under padding mode, each expert still computes on [B, S]
    actual_tokens = B * S
    # Effective ratio
    eff_ratio = avg_tokens_per_expert / actual_tokens
    # Total FLOPs (only counting the attention score matmul)
    flops_padding = E * B * n_heads * S * S * d * 2
    flops_useful = E * avg_tokens_per_expert * n_heads * S * d * 2
    return eff_ratio, flops_padding, flops_useful

print(f"{'config':>30}  {'valid token ratio':>18}  {'waste factor':>12}")
print("-" * 65)
configs = [
    ("dense (E=1, k=1)", 1, 4096, 1, 1, 32, 128),
    ("MoE E=8, k=2", 1, 4096, 8, 2, 32, 128),
    ("MoE E=64, k=1", 1, 4096, 64, 1, 32, 128),
    ("MoE E=64, k=2", 1, 4096, 64, 2, 32, 128),
    ("MoE E=128, k=2", 1, 4096, 128, 2, 32, 128),
]
for name, B, S, E, k, nh, d in configs:
    eff, _, _ = moe_padding_waste(B, S, E, k, nh, d)
    waste = 1 / eff if eff > 0 else float('inf')
    print(f"{name:>30}  {eff:>14.1%}  {waste:>9.1f}x")

print()
print("At E=128, k=2, the effective compute under padding mode is only 1.6% — 64x waste.")
print("The varlen mode sidesteps this problem entirely.")


## Appendix: The Evolution of FlashAttention

### A7.1 FlashAttention-2: Fewer Non-matmul FLOPs

FlashAttention-1's forward pass was already fast, but the backward pass and work partitioning still had room for improvement. FlashAttention-2 (Dao 2023) targets three optimizations.

First, **fewer non-matmul FLOPs**. On a GPU, matmul (tensor core) throughput is far higher than that of other operations (such as rescaling and softmax). FA-1 had some non-matmul operations in the inner loop; FA-2 rearranges them so that tensor core utilization is higher.

Second, **better parallelization**. FA-1 parallelizes mainly across the batch and head dimensions, leaving the GPU underutilized when sequences are long but the batch is small. FA-2 also splits the sequence dimension — different Q blocks are assigned to different SMs, significantly improving throughput for long-sequence scenarios.

Third, **work partitioning**. FA-1's inner/outer loop partitioning forced frequent synchronization between warps on the GPU. FA-2 redesigns this so each warp processes a block independently, reducing synchronization overhead.

In practice, FA-2 is about 2x faster than FA-1, reaching 50-70% of the theoretical matmul throughput on the A100.

### A7.2 FlashAttention-3: Hopper Architecture and FP8

FlashAttention-3 (Shah et al. 2024) is designed specifically for the Hopper architecture (H100), taking advantage of three new features.

First, **WGMMA (warp-group matrix multiply accumulate)**. An asynchronous tensor core instruction introduced in Hopper, allowing matmul to overlap with other operations. FA-3 schedules softmax and rescaling into the gaps of the matmul, almost completely hiding the non-matmul overhead.

Second, **TMA (tensor memory accelerator)**. Hopper's hardware DMA can move data asynchronously between HBM and SRAM without occupying the SM. FA-3 uses TMA for double buffering — the next K/V block is prefetched while the current block is being computed.

Third, **FP8 tensor cores**. Hopper supports FP8 (E4M3 / E5M2) tensor core matmul, with twice the throughput of FP16. FA-3 computes $QK^\top$ and $PV$ in FP8 in the forward pass, but still accumulates the softmax in FP32 to preserve numerical stability.

In practice, FA-3 reaches 1.5-2 PFLOPs (FP8) on the H100, approaching 75% of the hardware's theoretical upper bound.

### A7.3 A Guide to the Triton Implementation

OpenAI's Triton is a high-level language for writing GPU kernels in Python, more readable than CUDA. The FlashAttention project maintains an official Triton implementation, which is the best reference for learning about tiled kernels.

The source lives in the `flash_attn/flash_attn_triton.py` file of the [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) repository. The core structure is a function decorated with `@triton.jit`; the outer Q block loop uses `tl.range`, the inner K/V block loop is also `tl.range`, and each block is read with `tl.load` from HBM and written back with `tl.store`. The online softmax updates of $(m, d, o)$ translate directly into a few lines of Triton code, exactly mirroring the PyTorch version implemented in this appendix.

Reading advice: first build intuition from the PyTorch implementation in this appendix, then cross-reference the Triton version to understand the GPU programming details (shared memory, warp synchronization, block pointer advancement). The official Triton tutorial also has a simplified [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) version that is more readable than the Dao official version.


### References

- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018 (the original online softmax paper)
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), 2019


## Summary

- [ ] A GPU has two tiers of memory: HBM is large but slow (~2 TB/s), SRAM is small but fast (~19 TB/s); attention is the prototypical memory-bound operator
- [ ] Standard attention writes each $N \times N$ intermediate matrix to HBM 4-5 times; once sequences grow long, the transfer volume far exceeds the compute volume
- [ ] Naive softmax overflows (under FP32, $e^{88}$ is close to inf); stable softmax that subtracts the max is the baseline defense
- [ ] The logsumexp trick turns $\log \sum e^{x_i}$ into $x_\max + \log \sum e^{x_i - x_\max}$, avoiding both $\log(0)$ and $\exp$ overflow
- [ ] Online softmax fuses "find max" and "sum" into a single pass, using running $(m, d)$ and the $e^{m_\text{old} - m_\text{new}}$ rescale to produce an equivalent computation
- [ ] The attention output $o = \sum p_i v_i$ can likewise be accumulated in a single pass via the running $o_k$, with a rule symmetric to that of $d$
- [ ] Tiling splits Q, K, V into blocks that fit in SRAM; block-level online softmax stays correct because the update rule is associative
- [ ] FlashAttention is exact, not approximate — it merely reorders the computation, which is fundamentally different from sparse / linear attention
- [ ] Under MoE the attention itself is unchanged, but expert routing produces many small batches; padding mode wastes heavily, while the varlen mode is far better
- [ ] FlashAttention-2 optimizes work partitioning; FlashAttention-3 takes advantage of Hopper's WGMMA / TMA / FP8


The following exercises reinforce Online Softmax and Attention update rules.


### Exercise 1: Update the Running Maximum

Hint: the new maximum is the larger of the old maximum and current value. Rescale the old denominator by $e^{m_{old}-m_{new}}$, then add $e^{x_i-m_{new}}$.


In [ ]:
def online_softmax_homework(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        xi = xi.item()
        # TODO: complete the three lines
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        m = m_new
    return torch.tensor([math.exp(xi - m) / d for xi in x])

x = torch.tensor([1.0, 5.0, 3.0, 7.0, 2.0])
result = online_softmax_homework(x)
expected = torch.softmax(x, dim=0)
assert torch.allclose(result, expected, atol=1e-6), f"Difference too large: {(result - expected).abs().max()}"
print("Exercise 1 passed: online softmax is implemented correctly")


### Exercise 2: Verify Online Attention Numerically

Hint: update $o$ symmetrically with $d$: rescale its old value, then add $e^{x_i-m_{new}}v_i$.


In [ ]:
def online_attention_homework(q, K, V):
    """Attention for a single query."""
    d_head = q.shape[0]
    m = float('-inf')
    d = 0.0
    o = torch.zeros_like(q)

    for i in range(K.shape[0]):
        x_i = (q * K[i]).sum().item() / math.sqrt(d_head)
        v_i = V[i]
        # TODO: complete the four updates for m_new, rescale, d, and o
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        # 4. o = ?
        m = m_new

    return o / d

torch.manual_seed(0)
d = 8
N = 16
q = torch.randn(d)
K = torch.randn(N, d)
V = torch.randn(N, d)
out = online_attention_homework(q, K, V)
ref = F.scaled_dot_product_attention(q[None, None], K[None], V[None])[0, 0]
assert torch.allclose(out, ref, atol=1e-5), f"Difference: {(out - ref).abs().max()}"
print("Exercise 2 passed: online attention is numerically equivalent")


### Exercise 3: Estimate MoE Padding Overhead

Hint: with top-k routing, each expert receives on average $BSk/E$ tokens. A padded implementation processes a full $BS$ shape per expert; efficiency is the ratio. Variable-length execution concatenates only valid tokens, so speedup is approximately the inverse efficiency.


In [ ]:
B, S, E, k = 4, 8192, 64, 1
total_tokens = B * S
# TODO: complete the three lines
# tokens_per_expert = ?
# efficiency = ?       # useful / total work
# speedup_varlen = ?   # speedup of varlen over padding

assert tokens_per_expert == 512, f"tokens_per_expert should be 512; you got {tokens_per_expert}"
assert abs(efficiency - 1/64) < 1e-6, f"efficiency should be 1/64"
assert abs(speedup_varlen - 64) < 1e-6, f"speedup_varlen should be 64"
print("Exercise 3 passed: you understand wasted computation from MoE padding")
